<a href="https://colab.research.google.com/github/MithunSrinivas28/wafer-defect-ai/blob/main/Wafer_detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Mount Google Drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


### Load Dataset Using TensorFlow

In [3]:
TRAIN_PATH = "/content/drive/Shared-With-Me/Datasets/train"
TEST_PATH  = "/content/drive/Shared-With-Me/Datasets/test"


In [4]:
import tensorflow as tf

TRAIN_PATH = "/content/drive/MyDrive/Datasets/train"
TEST_PATH  = "/content/drive/MyDrive/Datasets/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_data = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)


Found 806 files belonging to 8 classes.
Found 105 files belonging to 8 classes.


In [5]:
import tensorflow as tf

# Reload only to get class names
temp_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/drive/MyDrive/Datasets/train",
    image_size=(224,224),
    batch_size=32
)

class_names = temp_ds.class_names
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("Num classes:", NUM_CLASSES)


Found 806 files belonging to 8 classes.
Classes: ['bridge', 'clean', 'cmp', 'crack', 'ler', 'open', 'others', 'vias']
Num classes: 8


In [6]:
import os

for c in os.listdir(TRAIN_PATH):
    print(c, len(os.listdir(TRAIN_PATH + "/" + c)))


bridge 136
clean 96
cmp 76
crack 105
ler 112
open 124
vias 93
others 64


### Normalize + Add Data Augmentation

In [7]:
from tensorflow.keras import layers

# Normalize (0–255 → 0–1)
normalization = layers.Rescaling(1./255)

# Data Augmentation
#data_augmentation = tf.keras.Sequential([
 #   layers.RandomFlip("horizontal"),
   # layers.RandomRotation(0.2),
   # layers.RandomZoom(0.2),
   # layers.RandomContrast(0.2),
#])
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
])

# Apply to datasets
train_data = train_data.map(lambda x, y: (normalization(data_augmentation(x)), y))
test_data  = test_data.map(lambda x, y: (normalization(x), y))


### Build the MobileNet Model (Your AI Brain)

In [8]:
from tensorflow.keras import layers, models
import tensorflow as tf

base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(224,224,3),
    include_top=False,
    #weights="imagenet"
    weights=None
  )

base_model.trainable = True

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    #layers.Dense(128, activation="relu"),
    #layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │         4,616 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 943,736 (3.60 MB)

 Trainable params: 931,624 (3.55 MB)

 Non-trainable params: 12,112 (47.31 KB)

##** Compile the Model**

In [9]:
model.compile(
    #optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    #loss="sparse_categorical_crossentropy",
   optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

y = []
for _, labels in train_data:
    y.extend(labels.numpy())

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)

class_weights = dict(enumerate(class_weights))
print(class_weights)


{0: np.float64(0.7408088235294118), 1: np.float64(1.0494791666666667), 2: np.float64(1.325657894736842), 3: np.float64(0.9595238095238096), 4: np.float64(0.8995535714285714), 5: np.float64(0.8125), 6: np.float64(1.57421875), 7: np.float64(1.0833333333333333)}


### Train the Model

In [11]:
EPOCHS = 15   # good for small dataset

history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=EPOCHS,
    class_weight=class_weights
)


Epoch 1/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 128s 3s/step - accuracy: 0.3374 - loss: 1.7254 - val_accuracy: 0.0667 - val_loss: 2.0830
Epoch 2/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 158ms/step - accuracy: 0.5553 - loss: 1.1868 - val_accuracy: 0.0667 - val_loss: 2.0843
Epoch 3/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 159ms/step - accuracy: 0.6433 - loss: 0.9162 - val_accuracy: 0.0667 - val_loss: 2.0881
Epoch 4/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 6s 215ms/step - accuracy: 0.6655 - loss: 0.9511 - val_accuracy: 0.0667 - val_loss: 2.0900
Epoch 5/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 159ms/step - accuracy: 0.7345 - loss: 0.7336 - val_accuracy: 0.0667 - val_loss: 2.0837
Epoch 6/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 7s 229ms/step - accuracy: 0.7956 - loss: 0.6138 - val_accuracy: 0.0667 - val_loss: 2.0858
Epoch 7/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 160ms/step - accuracy: 0.8010 - loss: 0.5666 - val_accuracy: 0.0667 - val_loss: 2.0865
Epoch 8/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 5s 207ms/step - accuracy: 0.8583 - loss: 0.3988 - val_accuracy: 0.0

In [12]:
model.evaluate(train_data)
model.evaluate(test_data)
#model is overfitting

26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 300ms/step - accuracy: 0.1512 - loss: 2.0686
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.1985 - loss: 2.0628


[2.0847110748291016, 0.12380952388048172]

In [13]:
model.save("/content/drive/MyDrive/wafer_model.keras")
print("Model saved!")


Model saved!
